# Turkish Content Moderation — Zero-shot Baseline

This pilot study compares a multilingual NLI-based zero-shot classifier with human annotations on 60 synthetic Turkish comments. It evaluates a classification baseline, not a generative LLM.

Run the cells in order in Google Colab and upload the latest reviewed `Label.xlsx` when prompted. A GPU is optional. The first model download requires internet access; runtime availability and speed may vary. This is a clean rerun notebook. Archived outputs from the completed baseline run are available in results/nli_baseline/.

## Annotation policy

- **Safe:** ordinary conversation, questions, compliments, and criticism or mild sarcasm without explicit insults, degradation, harassment, or threats.
- **Harmful:** explicit insults, degrading language, harassment, or threats. Under this project's policy, an explicit insult directed at someone's comment can also qualify; targeting a comment does not automatically make it safe.
- **Ambiguous:** both harmful and harmless interpretations are plausible, and missing context prevents a reliable decision. Sarcasm alone, uncertainty about the policy, or a low model score is not sufficient.

The reviewed examples include “Müthiş iş çıkarmışsın(!)” as Safe and “Gerçekten aptalca bir yorum” as Harmful. These document annotation decisions only; they are not supplied to the model as demonstrations.

All labels in the uploaded workbook are preserved. Here, “original human label” means the label as uploaded for this run, including any prior human review. It does not mean the first annotation version. The file hash identifies the version used.

The comments and model hypotheses remain in Turkish because the task evaluates Turkish content. Explanations, charts, and report headings are in English.


In [ ]:
%pip -q install "transformers==4.57.1" sentencepiece protobuf pandas openpyxl scikit-learn matplotlib huggingface_hub


## 1. Upload the reviewed Label.xlsx
The expected layout is Sheet1, headers in B2:D2, and 60 records beneath them. This cell validates the input without correcting labels or silently dropping rows. Unexpected data stops execution.

In [ ]:
import io, json, hashlib, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd
from google.colab import files
from IPython.display import display

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError('Upload only the Label.xlsx file.')
filename, payload = next(iter(uploaded.items()))
df = pd.read_excel(io.BytesIO(payload), sheet_name='Sheet1', header=1, usecols='B:D', keep_default_na=False)
if list(df.columns) != ['ID', 'text', 'human_label']:
    raise ValueError('Expected headers in Sheet1!B2:D2: ID, text, human_label')
LABELS = ['Safe', 'Harmful', 'Ambiguous']
if len(df) != 60 or df['ID'].tolist() != list(range(1,61)):
    raise ValueError('IDs must be exactly 1–60 in order; check the workbook.')
if not df['text'].map(lambda x: isinstance(x,str) and bool(x.strip())).all():
    raise ValueError('A comment is blank or is not valid text.')
if not df['human_label'].isin(LABELS).all():
    display(df.loc[~df['human_label'].isin(LABELS)])
    raise ValueError('Labels were not changed. Check missing or invalid values.')
if df['text'].str.strip().str.casefold().duplicated().any():
    raise ValueError('Duplicate text found; review it before evaluation.')
original_labels = df['human_label'].copy(deep=True)
dataset_hash = hashlib.sha256(payload).hexdigest()
display(df['human_label'].value_counts().rename_axis('label').to_frame('count'))
print('Validated:', len(df), 'records. Labels were not changed.')


## 2. Load the model and define candidate classes
The Turkish class descriptions are a simple baseline approximation of the annotation policy; they do not encode every policy detail. If you change them after inspecting results, record a separate experiment and use new independent examples for an unbiased evaluation.

The model revision and dataset hash are recorded in the run manifest. The model revision is pinned to the completed baseline run. Package and hardware differences may still affect reproducibility.

In [ ]:
import torch
from transformers import pipeline, set_seed

set_seed(42)
MODEL_ID = 'MoritzLaurer/mDeBERTa-v3-base-mnli-xnli'
REVISION = '8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c'
DESCRIPTIONS = {
    'Safe': 'zararsız bir konuşma, soru, iltifat veya kişiyi aşağılamayan eleştiri',
    'Harmful': 'açık bir hakaret, kişiyi aşağılama, taciz veya zarar tehdidi',
    'Ambiguous': 'zararlı mı zararsız mı olduğu eksik bağlam nedeniyle belirsiz bir ifade',
}
TEMPLATE = 'Bu metin {} içeriyor.'
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline('zero-shot-classification', model=MODEL_ID,
                      revision=REVISION, device=device, dtype=torch.float32)
print('Model:', MODEL_ID, 'Revision:', REVISION, 'Device:', device)


## 3. Generate 60 predictions
Only comment text is passed to the classifier. Human labels are used afterwards for comparison. The selected label has the highest relative score among the three candidate hypotheses. These scores are not calibrated probabilities of correctness, and a low score does not automatically imply Ambiguous.

In [ ]:
from tqdm.auto import tqdm
reverse = {v:k for k,v in DESCRIPTIONS.items()}
predictions = []
for text in tqdm(df['text'].tolist()):
    result = classifier(text, candidate_labels=list(DESCRIPTIONS.values()),
                        hypothesis_template=TEMPLATE, multi_label=False)
    scores = {reverse[k]:float(v) for k,v in zip(result['labels'],result['scores'])}
    top = reverse[result['labels'][0]]
    predictions.append({'model_label':top, 'model_score':scores[top],
                        **{f'score_{k}':scores[k] for k in LABELS}})
results = pd.concat([df.copy(), pd.DataFrame(predictions)], axis=1)
assert results['human_label'].equals(original_labels)
assert results['model_label'].isin(LABELS).all() and len(results) == 60
results['agreement'] = results['human_label'] == results['model_label']
display(results.head())


## 4. Evaluate agreement and inspect disagreements
Accuracy measures agreement with the uploaded human labels, which are not assumed to be infallible. Review-flagged examples are not removed. Do not change labels merely to match model predictions. Use the exported `review_reason` field to document model errors, annotation-policy issues, or unresolved context.

Inspect macro-F1 and per-class precision and recall alongside accuracy. The majority-class baseline is calculated from the uploaded data. Undefined precision or recall is reported as zero (`zero_division=0`); inspect class support and predicted counts when interpreting those values.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

y_true, y_pred = results['human_label'], results['model_label']
metrics = {
    'n':len(results),
    'agreement_count':int(results['agreement'].sum()),
    'accuracy_vs_original_human':accuracy_score(y_true,y_pred),
    'macro_f1_vs_original_human':f1_score(y_true,y_pred,labels=LABELS,average='macro',zero_division=0),
    'majority_baseline_accuracy':float(y_true.value_counts().max()/len(y_true)),
}
report = pd.DataFrame(classification_report(y_true,y_pred,labels=LABELS,output_dict=True,zero_division=0)).T
cm = confusion_matrix(y_true,y_pred,labels=LABELS)
display(pd.Series(metrics).to_frame('value'))
display(report)
fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(cm,display_labels=LABELS).plot(ax=ax,cmap='Blues',colorbar=False)
ax.set_xlabel('Model prediction'); ax.set_ylabel('Uploaded human label')
ax.set_title('60 synthetic Turkish comments')
fig.tight_layout()
disagreements = results.loc[~results['agreement']].copy()
disagreements['review_reason'] = ''
display(disagreements)


## 5. Download the results
The ZIP contains CSV files that can be opened in Excel, a confusion matrix image, metrics, and a run manifest. Confusion matrix rows represent human labels; columns represent model predictions. The manifest records the model revision, candidate descriptions, package versions, and dataset hash.

In [ ]:
import zipfile
OUT = Path('moderation_results')
OUT.mkdir(exist_ok=True)
results.to_csv(OUT/'predictions.csv',index=False,encoding='utf-8-sig')
disagreements.to_csv(OUT/'disagreements.csv',index=False,encoding='utf-8-sig')
report.to_csv(OUT/'classification_report.csv',encoding='utf-8-sig')
pd.DataFrame(cm,index=LABELS,columns=LABELS).rename_axis('human_label').to_csv(OUT/'confusion_matrix.csv',encoding='utf-8-sig')
fig.savefig(OUT/'confusion_matrix.png',dpi=160,bbox_inches='tight')
(OUT/'metrics.json').write_text(json.dumps(metrics,ensure_ascii=False,indent=2),encoding='utf-8')
manifest = {
    'created_utc':datetime.now(timezone.utc).isoformat(),
    'model_id':MODEL_ID,'revision':REVISION,'dataset_sha256':dataset_hash,
    'source_filename':filename,'candidate_descriptions':DESCRIPTIONS,
    'hypothesis_template':TEMPLATE,'multi_label':False,'seed':42,
    'device':str(classifier.device),'python':platform.python_version(),
    'packages':{p:importlib.metadata.version(p) for p in ['transformers','torch','pandas','scikit-learn','huggingface_hub','sentencepiece','openpyxl']},
    'reference':'original human_label; no edits',
    'limitations':'60 synthetic samples; one annotator; relative uncalibrated scores; NLI model, not generative LLM',
}
(OUT/'run_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
bundle = Path('moderation_results.zip')
with zipfile.ZipFile(bundle,'w',zipfile.ZIP_DEFLATED) as z:
    for name in ['predictions.csv','disagreements.csv','classification_report.csv','confusion_matrix.csv','confusion_matrix.png','metrics.json','run_manifest.json']:
        z.write(OUT/name,arcname=name)
files.download(str(bundle))


## Interpret and report the results
After execution, report the sample size, accuracy, macro-F1, and especially Ambiguous recall. Review up to 10–15 disagreements, covering sarcasm, implicit threats, insulting criticism, and annotation-policy uncertainty. Distinguish model errors from unresolved annotation decisions and support each conclusion with examples.

This is a small pilot using synthetic data and one human annotator, with labels reviewed before inference. It does not establish performance on real social-media content. Repeatedly tuning the hypotheses on these 60 examples turns them into development data; a later evaluation would require independent examples.

Sources: [Model card](https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli), [Transformers pipeline documentation](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/pipelines#transformers.ZeroShotClassificationPipeline).
